# Project FORESIGHT — Phase 7: Baseline Demand Forecasting

**Input:** `data/processed/features/forecast_features.parquet` (Phase 6)  
**Target:** `units_sold`  
**Grain:** `date + source_dataset + entity_id + product_key`  

Baselines only: Naive, Seasonal Naive, Moving Average (7/14/30), Historical Mean.  
**No ML models in this phase.**


## 1–2. Load Phase 6 Features

In [1]:
import os, sys
import numpy as np
import pandas as pd

BASE_DIR = os.path.abspath(".")
if os.path.basename(BASE_DIR) == "notebooks":
    BASE_DIR = os.path.abspath("..")
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

from src.baseline_forecasting import (
    load_features, create_time_split, confirm_seasonal_period,
    generate_all_predictions, evaluate_baselines, best_baselines,
    high_value_sku_analysis, save_baseline_results, create_forecast_charts,
    create_comparison_chart, write_baseline_report, SEASONAL_PERIOD, MODEL_COLS,
)
from src.validate_baselines import run_validation

df = load_features()
print("rows", len(df), "cols", df.shape[1])
print("date", df.date.min().date(), "->", df.date.max().date())
print("sources", df.source_dataset.value_counts().to_dict())
print("target nulls", int(df.units_sold.isna().sum()))
print("columns sample:", list(df.columns)[:15], "...")


rows 1995496 cols 62
date 2009-12-01 -> 2025-12-31
sources {'SYNTHETIC': 1461000, 'UCI': 534496}
target nulls 0
columns sample: ['date', 'source_dataset', 'entity_id', 'entity_type', 'product_key', 'sku_id', 'units_sold', 'revenue', 'average_unit_price', 'transaction_count', 'unique_customers', 'promotion_flag', 'year', 'month', 'quarter'] ...


## 3–4. Validation of grain & target

In [2]:
assert set(["date","source_dataset","entity_id","product_key","units_sold","split"]).issubset(df.columns)
assert df.duplicated(["date","source_dataset","entity_id","product_key"]).sum() == 0
print("Forecast grain OK; target=units_sold; duplicate keys=0")
print("UCI entities", df[df.source_dataset=="UCI"].entity_id.nunique(),
      "SYN entities", df[df.source_dataset=="SYNTHETIC"].entity_id.nunique())


Forecast grain OK; target=units_sold; duplicate keys=0


UCI entities 1 SYN entities 10


## 5. Target Analysis

In [3]:
for src in sorted(df.source_dataset.unique()):
    sub = df[df.source_dataset==src]
    print(src, "mean", round(sub.units_sold.mean(),3),
          "median", round(sub.units_sold.median(),3),
          "zero_pct", round(100*(sub.units_sold==0).mean(),2),
          "p95", round(sub.units_sold.quantile(0.95),2))


SYNTHETIC mean 7.389 median 0.0 zero_pct 62.67 p95 33.0


UCI mean 21.433 median 6.0 zero_pct 0.0 p95 73.0


## 6. Chronological Split

In [4]:
split_summary = create_time_split(df)
print(split_summary.to_string(index=False))
for src in sorted(df.source_dataset.unique()):
    for sp in ["train","validation","test"]:
        sub = df[(df.source_dataset==src)&(df.split==sp)]
        print(f"{src}/{sp}: {sub.date.min().date()} -> {sub.date.max().date()} rows={len(sub):,}")


source_dataset      split start_date   end_date    rows  unique_entities  unique_products
     SYNTHETIC      train 2022-01-01 2025-03-13 1168000               10              100
     SYNTHETIC validation 2025-03-14 2025-08-06  146000               10              100
     SYNTHETIC       test 2025-08-07 2025-12-31  147000               10              100
           UCI      train 2009-12-01 2011-07-13  401604                1             4657
           UCI validation 2011-07-14 2011-09-25   53174                1             3036
           UCI       test 2011-09-26 2011-12-09   79718                1             3195


SYNTHETIC/train: 2022-01-01 -> 2025-03-13 rows=1,168,000
SYNTHETIC/validation: 2025-03-14 -> 2025-08-06 rows=146,000


SYNTHETIC/test: 2025-08-07 -> 2025-12-31 rows=147,000
UCI/train: 2009-12-01 -> 2011-07-13 rows=401,604
UCI/validation: 2011-07-14 -> 2011-09-25 rows=53,174


UCI/test: 2011-09-26 -> 2011-12-09 rows=79,718


## 7–10. Generate Baselines (Naive / Seasonal Naive / MA / Historical Mean)

In [5]:
seasonality = confirm_seasonal_period(df)
print("Seasonal period:", seasonality["selected_period"])
print("Rationale:", seasonality["rationale"])
print("DOW CV:", {k: seasonality[k] for k in seasonality if k.endswith("_dow_cv")})

pred_df = generate_all_predictions(df)
print("Prediction columns:", list(MODEL_COLS.values()))
print(pred_df[["date","source_dataset","entity_id","product_key","units_sold","split"] + list(MODEL_COLS.values())].head(3).to_string(index=False))


Seasonal period: 7
Rationale: Phase 5 EDA (§8 Seasonality) found clear day-of-week effects for both SYNTHETIC (weekend lift) and UCI (weekday wholesale peaks). A 7-day seasonal naive aligns forecasts with the same weekday.
DOW CV: {'SYNTHETIC_dow_cv': 0.2241, 'UCI_dow_cv': 0.173}


Prediction columns: ['pred_naive', 'pred_seasonal_naive', 'pred_ma_7', 'pred_ma_14', 'pred_ma_30', 'pred_historical_mean']
      date source_dataset entity_id product_key  units_sold split  pred_naive  pred_seasonal_naive  pred_ma_7  pred_ma_14  pred_ma_30  pred_historical_mean
2009-12-01            UCI    ONLINE   UCI_10002          12 train         NaN                  NaN        NaN         NaN         NaN                   NaN
2009-12-01            UCI    ONLINE   UCI_10120          60 train         NaN                  NaN        NaN         NaN         NaN                   NaN
2009-12-01            UCI    ONLINE  UCI_10123C           3 train         NaN                  NaN        NaN         NaN         NaN                   NaN


## 11–15. Metric Evaluation (UCI / Synthetic / Product / Store)

In [6]:
tables = evaluate_baselines(pred_df)
best = best_baselines(tables["comparison"])
best_models = {s: i["model"] for s,i in best.items()}
tables["high_value"] = high_value_sku_analysis(df, tables["by_product"], best_models)

print("=== TEST metrics by source ===")
test_m = tables["by_source"][tables["by_source"].split=="test"].sort_values(["source_dataset","WAPE"])
print(test_m[["source_dataset","model","MAE","RMSE","MAPE","sMAPE","WAPE","n"]].to_string(index=False))

print("\n=== Best baselines ===")
for src, info in best.items():
    print(src, info)

print("\n=== High-value SKU analysis ===")
print(tables["high_value"].to_string(index=False))

print("\n=== Synthetic store WAPE (best model) ===")
be = tables["by_entity"]
syn_model = best["SYNTHETIC"]["model"]
print(be[(be.source_dataset=="SYNTHETIC")&(be.model==syn_model)][["entity_id","MAE","RMSE","WAPE"]].sort_values("WAPE").to_string(index=False))


=== TEST metrics by source ===
source_dataset             model     MAE    RMSE     MAPE    sMAPE     WAPE      n
     SYNTHETIC             naive  5.2717 10.4688  97.0020  45.1750  72.8181 147000
     SYNTHETIC   historical_mean  7.3879 10.0161  58.6245 150.1693 102.0497 147000
     SYNTHETIC moving_average_30  7.4157 10.1194  59.8022 150.7727 102.4335 147000
     SYNTHETIC  moving_average_7  7.4786 11.0391  69.9000 114.9469 103.3033 147000
     SYNTHETIC moving_average_14  7.8060 10.8799  64.5076 141.2780 107.8252 147000
     SYNTHETIC    seasonal_naive  9.1952 14.9194  83.9620  89.5054 127.0150 147000
           UCI moving_average_30 18.8542 72.0799 317.0346  84.3796  86.3870  79545
           UCI moving_average_14 19.0853 72.6680 309.3890  83.0470  87.4460  79545
           UCI  moving_average_7 19.5614 74.2491 301.1717  82.4668  89.6276  79545
           UCI   historical_mean 20.5473 75.7927 379.6150  89.2190  94.1446  79545
           UCI             naive 23.6822 99.8604 313.511

## 16–17. Visualizations & Comparison

In [7]:
paths = save_baseline_results(pred_df, tables)
chart_paths = create_forecast_charts(pred_df)
chart_paths.append(create_comparison_chart(tables["comparison"]))
print("Saved files:")
for k,v in paths.items():
    print(" ", k, "->", v)
print("Charts:")
for p in chart_paths:
    print(" ", p)
print("\nComparison (TEST):")
print(tables["comparison"][["source_dataset","rank","model","MAE","RMSE","sMAPE","WAPE"]].to_string(index=False))


Saved files:
  predictions -> C:\Users\SURAG\Documents\zidio\Project_FORESIGHT\Demand-Inventory-Intelligence\data\processed\forecasts\baseline\baseline_predictions.parquet
  metrics -> C:\Users\SURAG\Documents\zidio\Project_FORESIGHT\Demand-Inventory-Intelligence\data\processed\forecasts\baseline\baseline_metrics.parquet
  by_source -> C:\Users\SURAG\Documents\zidio\Project_FORESIGHT\Demand-Inventory-Intelligence\data\processed\forecasts\baseline\baseline_metrics_by_source.parquet
  by_product -> C:\Users\SURAG\Documents\zidio\Project_FORESIGHT\Demand-Inventory-Intelligence\data\processed\forecasts\baseline\baseline_metrics_by_product.parquet
  by_entity -> C:\Users\SURAG\Documents\zidio\Project_FORESIGHT\Demand-Inventory-Intelligence\data\processed\forecasts\baseline\baseline_metrics_by_entity.parquet
  comparison -> C:\Users\SURAG\Documents\zidio\Project_FORESIGHT\Demand-Inventory-Intelligence\data\processed\forecasts\baseline\baseline_comparison.parquet
  high_value -> C:\Users\SURA

## 18–19. Business Interpretation & Phase 8 Recommendations

In [8]:
for src, info in best.items():
    print(f"OBSERVATION: Best {src} baseline is {info['model']}.")
    print(f"EVIDENCE: TEST WAPE={info['WAPE']:.4f}, MAE={info['MAE']:.4f}, RMSE={info['RMSE']:.4f}, sMAPE={info['sMAPE']:.4f}")
    print("BUSINESS INTERPRETATION: Source-specific demand regimes require separate benchmarks.")
    print(f"IMPLICATION FOR ML: Phase 8 must beat {src} WAPE={info['WAPE']:.4f} on the same TEST window.\n")


OBSERVATION: Best SYNTHETIC baseline is naive.
EVIDENCE: TEST WAPE=72.8181, MAE=5.2717, RMSE=10.4688, sMAPE=45.1750
BUSINESS INTERPRETATION: Source-specific demand regimes require separate benchmarks.
IMPLICATION FOR ML: Phase 8 must beat SYNTHETIC WAPE=72.8181 on the same TEST window.

OBSERVATION: Best UCI baseline is moving_average_30.
EVIDENCE: TEST WAPE=86.3870, MAE=18.8542, RMSE=72.0799, sMAPE=84.3796
BUSINESS INTERPRETATION: Source-specific demand regimes require separate benchmarks.
IMPLICATION FOR ML: Phase 8 must beat UCI WAPE=86.3870 on the same TEST window.



## 20. Validation

In [9]:
result = run_validation()
print("VALIDATION:", result.summary())
assert result.failed == 0, f"Baseline validation failed: {result.failed}"
report_path = write_baseline_report(
    df, split_summary, seasonality, tables, best, chart_paths,
    validation_summary=result.summary(), paths=paths,
)
print("Report:", report_path)
print("Phase 7 COMPLETE — STOP before Phase 8.")


PHASE 7 BASELINE VALIDATION

[1] Phase 6 Prerequisite
  [+] Phase 6 forecast_features exists

[2] Output Files
  [+] predictions file exists -- C:\Users\SURAG\Documents\zidio\Project_FORESIGHT\Demand-Inventory-Intelligence\data\processed\forecasts\baseline\baseline_predictions.parquet
  [+] metrics file exists -- C:\Users\SURAG\Documents\zidio\Project_FORESIGHT\Demand-Inventory-Intelligence\data\processed\forecasts\baseline\baseline_metrics.parquet
  [+] by_source file exists -- C:\Users\SURAG\Documents\zidio\Project_FORESIGHT\Demand-Inventory-Intelligence\data\processed\forecasts\baseline\baseline_metrics_by_source.parquet
  [+] comparison file exists -- C:\Users\SURAG\Documents\zidio\Project_FORESIGHT\Demand-Inventory-Intelligence\data\processed\forecasts\baseline\baseline_comparison.parquet
  [+] Predictions readable -- 1,995,496 rows x 14 cols



[3] Forecast Grain
  [+] Grain column 'date' present
  [+] No null in 'date'
  [+] Grain column 'source_dataset' present
  [+] No null in 'source_dataset'
  [+] Grain column 'entity_id' present
  [+] No null in 'entity_id'
  [+] Grain column 'product_key' present
  [+] No null in 'product_key'


  [+] No duplicate forecast keys -- duplicates=0
  [+] Row count matches Phase 6 features -- pred=1,995,496 features=1,995,496

[4] Prediction Validity
  [+] Prediction column 'pred_naive' exists
  [+] naive: no infinite predictions -- inf=0
  [+] naive: no negative predictions -- neg=0
  [+] Prediction column 'pred_seasonal_naive' exists
  [+] seasonal_naive: no infinite predictions -- inf=0
  [+] seasonal_naive: no negative predictions -- neg=0
  [+] Prediction column 'pred_ma_7' exists
  [+] moving_average_7: no infinite predictions -- inf=0


  [+] moving_average_7: no negative predictions -- neg=0
  [+] Prediction column 'pred_ma_14' exists
  [+] moving_average_14: no infinite predictions -- inf=0
  [+] moving_average_14: no negative predictions -- neg=0
  [+] Prediction column 'pred_ma_30' exists
  [+] moving_average_30: no infinite predictions -- inf=0
  [+] moving_average_30: no negative predictions -- neg=0
  [+] Prediction column 'pred_historical_mean' exists
  [+] historical_mean: no infinite predictions -- inf=0
  [+] historical_mean: no negative predictions -- neg=0

[5] Leakage Checks


  [+] naive: first-row prediction is NaN (no prior history) -- non-null=0
  [+] seasonal_naive: first-row prediction is NaN (no prior history) -- non-null=0
  [+] moving_average_7: first-row prediction is NaN (no prior history) -- non-null=0
  [+] moving_average_14: first-row prediction is NaN (no prior history) -- non-null=0
  [+] moving_average_30: first-row prediction is NaN (no prior history) -- non-null=0
  [+] historical_mean: first-row prediction is NaN (no prior history) -- non-null=0
  [+] Naive equals Actual(t-1)
  [+] Seasonal naive equals Actual(t-7)
  [+] MA-7 excludes current target -- expected=16.285714 actual=16.285714
  [+] Historical mean uses only past observations -- expected=18.533333 actual=18.533333

[6] Source Separation
  [+] Two sources present -- sources=['SYNTHETIC', 'UCI']


  [+] No shared entity_id across sources
  [+] No shared product_key across sources

[7] Chronological Split
  [+] split column present


  [+] SYNTHETIC: train_end < val_start -- 2025-03-13 < 2025-03-14
  [+] SYNTHETIC: val_end < test_start -- 2025-08-06 < 2025-08-07
  [+] UCI: train_end < val_start -- 2011-07-13 < 2011-07-14
  [+] UCI: val_end < test_start -- 2011-09-25 < 2011-09-26

[8] Metrics Validity
  [+] by_source metrics readable -- 24 rows
  [+] Metric column MAE present
  [+] MAE finite
  [+] MAE non-negative
  [+] Metric column RMSE present
  [+] RMSE finite
  [+] RMSE non-negative
  [+] Metric column sMAPE present
  [+] sMAPE finite
  [+] sMAPE non-negative
  [+] Metric column WAPE present
  [+] WAPE finite
  [+] WAPE non-negative
  [+] Stored UCI naive TEST MAE matches recalculation -- stored=23.6822 recalc=23.6822
  [+] Comparison file has both sources
  [+] UCI: comparison has ranks
  [+] SYNTHETIC: comparison has ranks

VALIDATION RESULT: 69/69 PASS
VALIDATION: 69/69 PASS


Report: C:\Users\SURAG\Documents\zidio\Project_FORESIGHT\Demand-Inventory-Intelligence\docs\baseline_forecasting_report.md
Phase 7 COMPLETE — STOP before Phase 8.


## 21. Summary

In [10]:
print("Input rows:", len(df))
print("Models:", list(MODEL_COLS))
print("Seasonal period:", SEASONAL_PERIOD)
print("Best:", best)
print("Validation:", result.summary())


Input rows: 1995496
Models: ['naive', 'seasonal_naive', 'moving_average_7', 'moving_average_14', 'moving_average_30', 'historical_mean']
Seasonal period: 7
Best: {'SYNTHETIC': {'model': 'naive', 'MAE': 5.2717, 'RMSE': 10.4688, 'sMAPE': 45.175, 'WAPE': 72.8181, 'MAPE': 97.002, 'n': 147000}, 'UCI': {'model': 'moving_average_30', 'MAE': 18.8542, 'RMSE': 72.0799, 'sMAPE': 84.3796, 'WAPE': 86.387, 'MAPE': 317.0346, 'n': 79545}}
Validation: 69/69 PASS
